# RL_PCB: Reinforcement Learning for PCB Placement and Routing

This notebook allows you to run experiments from the RL_PCB project on Google Colab, leveraging external GPUs for faster training.

## 1. Setup Environment

First, we need to clone the repository and install all necessary dependencies, including building the C++ binaries for placement and routing.

In [ ]:
# @title Clone Repository and Install Dependencies
import os

# 1. Clone the repository (replace with your own fork if needed)
!git clone https://github.com/Alli-Ekundayo/RL_PCB
%cd RL_PCB

# 2. Install system dependencies
!sudo apt-get update
!sudo apt-get install -y build-essential cmake libboost-system-dev libboost-filesystem-dev libboost-program-options-dev libboost-test-dev

# 3. Build C++ binaries
!chmod +x install_tools.sh
!./install_tools.sh --update_utility_binaries --skip-repository-check

# 4. Install Python dependencies
!pip install -r requirements.txt
!pip install lib/pcb_netlist_graph-0.0.1-py3-none-any.whl
!pip install lib/pcb_file_io-0.0.1-py3-none-any.whl

# 5. Install JAX for real DreamerV3 support
!pip install jax jaxlib flax

# 6. Set Environment Variables
os.environ['RL_PCB'] = os.getcwd()
os.environ['PYTHONPATH'] = f"{os.environ.get('PYTHONPATH', '')}:{os.getcwd()}/src/training"

print("\nSetup Complete!")

## 2. Configure Experiment

Select your algorithm and hyperparameters here.

In [ ]:
# @title Select Experiment Settings
algorithm = "DreamerV3" # @param ["TD3", "SAC", "DreamerV3"]
device = "cuda" # @param ["cuda", "cpu"]
max_timesteps = 50000 # @param {type:"integer"}
experiment_name = "colab_test_run" # @param {type:"string"}

import json

# Create hyperparameters file
hp = {
    "learning_rate": 0.0003,
    "buffer_size": 100000,
    "batch_size": 256,
    "gamma": 0.99,
    "tau": 0.005
}

os.makedirs("hyperparameters", exist_ok=True)
hp_file = f"hyperparameters/hp_{algorithm.lower()}.json"
with open(hp_file, "w") as f:
    json.dump(hp, f)

print(f"Configured {algorithm} for {max_timesteps} steps on {device}.")

## 3. Run Training

Execute the training loop.

In [ ]:
# @title Start Training
work_dir = f"work/{experiment_name}"
os.makedirs(work_dir, exist_ok=True)

!python src/training/train.py \
    --policy {algorithm} \
    --max_timesteps {max_timesteps} \
    --training_pcb dataset/base/training.pcb \
    --evaluation_pcb dataset/base/evaluation.pcb \
    --tensorboard_dir {work_dir} \
    --device {device} \
    --experiment {experiment_name} \
    --hyperparameters {hp_file} \
    --verbose 1

## 4. Evaluation and Visualization

Generate reports and visualize the routed PCBs.

In [ ]:
# @title Generate Experiment Report
!mkdir -p reports

# 1. Generate report config
rc = {
    "charts": {
        experiment_name: {
            "experiments": [experiment_name],
            "algorithms": [algorithm],
            "multi_agent": True,
            "window": 100,
            "title": f"{algorithm} Training Performance",
            "xlabel": "Timesteps",
            "ylabel": "Reward",
            "label": {}
        }
    },
    "tables": {},
    "author": "Colab User",
    "timestamp": "2026-04-20"
}

with open("report_config.json", "w") as f:
    json.dump(rc, f)

# 2. Run report generation
!python src/report_generation/generate_experiment_report.py \
    --dirs {work_dir} \
    --report_config report_config.json \
    --output reports/experiment_report.pdf \
    --hyperparameters {hp_file} \
    -y